In [1]:
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS
import statsmodels.api as sm


FX_rates_df = pd.read_csv('../data/processed/FX_rates/Weekly_FX_Rates.csv', parse_dates=['Date'])
Oil_prices_df = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['Date'])
MSCI_df = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['Date'])

studycountries = [
    'Saudi Arabia', 'United Arab Emirates', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines',
    'Turkey', 'Chile', 'China', 'South Africa', 'South Korea', 'Thailand'
]

exporters = ['Saudi Arabia', 'United Arab Emirates', 'Qatar', 'Colombia', 'Brazil', 'Mexico', 'Egypt','Malaysia']

/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_61979/254922204.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Oil_prices_df = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['Date'])
/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_61979/254922204.py:9: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  MSCI_df = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['Date'])


In [2]:


# Melt and merge
fx_melt = FX_rates_df.melt(id_vars=['Date'], var_name='country', value_name='fx_rate')
msci_melt = MSCI_df.melt(id_vars=['Date'], var_name='country', value_name='msci_index')

df = fx_melt.merge(msci_melt, on=['Date', 'country'], how='inner')
df = df.merge(Oil_prices_df[['Date', 'Brent']], on='Date', how='inner')

# Filter
df = df[df['country'].isin(studycountries)].copy()

# --- Compute oil return CORRECTLY (unique by Date) ---
oil = Oil_prices_df[['Date', 'Brent']].drop_duplicates('Date').sort_values('Date').copy()
oil['dlog_brent'] = np.log(oil['Brent']).diff()
oil = oil.dropna(subset=['dlog_brent'])

# Merge oil returns into panel
df = df.merge(oil[['Date', 'dlog_brent']], on='Date', how='inner')

# Exporter dummy
df['exporter'] = df['country'].isin(exporters).astype(int)

# Sort before diffs
df = df.sort_values(['country', 'Date'])

# Panel diffs
df['dlog_msci'] = df.groupby('country')['msci_index'].transform(lambda s: np.log(s).diff())
df['dlog_fx']   = df.groupby('country')['fx_rate'].transform(lambda s: np.log(s).diff())

# Interaction
df['dlog_brent_x_exporter'] = df['dlog_brent'] * df['exporter']

# Drop missing
df = df.dropna(subset=['dlog_fx', 'dlog_brent', 'dlog_brent_x_exporter', 'dlog_msci'])
df = df.set_index(['country', 'Date']).sort_index()


# reggressions

In [4]:
# Regressors
X = df[['dlog_brent', 'dlog_brent_x_exporter']]
X = sm.add_constant(X)

y = df['dlog_fx']

model = PanelOLS(y, X, entity_effects=True, time_effects=False)

res1 = model.fit(cov_type='clustered', cluster_entity=True)

print(res1.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                dlog_fx   R-squared:                     7.706e-06
Estimator:                   PanelOLS   R-squared (Between):             -0.0066
No. Observations:               33330   R-squared (Within):            7.706e-06
Date:                Mon, Feb 09 2026   R-squared (Overall):           7.696e-06
Time:                        16:05:10   Log-likelihood                -1.583e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.1284
Entities:                          15   P-value                           0.8795
Avg Obs:                       2222.0   Distribution:                 F(2,33313)
Min Obs:                       2086.0                                           
Max Obs:                       2334.0   F-statistic (robust):             1.9827
                            

In [5]:
df['residual'] = res1.resids

# Oil moved, FX didn't
df['gap'] = -df['residual']

# Accumulate within country
df['CumulativeGap'] = df.groupby(level=0)['gap'].cumsum()


# Need log FX level for forward returns
df['log_fx'] = np.log(df['fx_rate'])

horizons = [4, 12, 24]  # weeks ahead (1m, 3m, 6m approx)

for k in horizons:
    df[f'fwd_dlog_fx_{k}'] = df.groupby(level=0)['log_fx'].shift(-k) - df['log_fx']


results_step2 = {}

for k in horizons:
    temp = df.dropna(subset=[f'fwd_dlog_fx_{k}', 'CumulativeGap'])

    y2 = temp[f'fwd_dlog_fx_{k}']
    X2 = sm.add_constant(temp[['CumulativeGap']])

    mod2 = PanelOLS(y2, X2, entity_effects=True)  # country FE only
    res2 = mod2.fit(cov_type='clustered', cluster_entity=True)

    results_step2[k] = res2
    print(f"\n===== Horizon {k} weeks =====")
    print(res2.summary)




===== Horizon 4 weeks =====
                          PanelOLS Estimation Summary                           
Dep. Variable:          fwd_dlog_fx_4   R-squared:                        0.1078
Estimator:                   PanelOLS   R-squared (Between):             -1409.3
No. Observations:               33270   R-squared (Within):               0.1078
Date:                Mon, Feb 09 2026   R-squared (Overall):              0.0952
Time:                        16:05:55   Log-likelihood                -2.805e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      4018.0
Entities:                          15   P-value                           0.0000
Avg Obs:                       2218.0   Distribution:                 F(1,33254)
Min Obs:                       2082.0                                           
Max Obs:                       2330.0   F-statistic (robust):             176.33